In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd

from pathlib import Path
from scipy.ndimage import gaussian_filter

# Data path
'/work/bb1555/user/kolja/datasets/c3sar_lindenberg/'

Koljas Readme:

# C3SAR shared database
a collection of 1D / 3D radiation simulations based on ICON simulations.

## c3sar_lindenberg
cases:
- 20260519: various different cloud situations with some simple period during afternoon, investigated by Jonas with Pyrnet
- 20260606: simple case, also sentinel snapshot available
- 20260607: simple case, also earthCARE overpass

experiments:
- exp023: former standart ICON experiment with dt_rad=5min, with ICON2024.04
- exp024: same configuration but run with ICON2026.04 version after moving from bb1376 to bb1555

- v00: 200x200 pixel, 10x1e8 photons / timestep, 21 steps a 30min resolution
- v01: 200x200 pixel, 10x1e8 photons / timestep, 41 steps a 15min resolution
    - similar to v00 but from the same run as the v02 high resolution dataset
- v02: 200x200 pixel, 10x1e8 photons / timestep, 25 steps a 5min resolution
    - from 12:00 to 14:00 UTC, for this period a 10s res cloud field is available, goal is comparison with pyrnet.



In [ ]:
path = '/work/bb1555/user/kolja/datasets/c3sar_lindenberg/c3sar_exp024_DOM03_v01_20260606_1D.nc'
ds = xr.load_dataset(path)
ds

In [ ]:
# Base directory
data_dir = Path(
    "/work/bb1555/user/kolja/datasets/c3sar_lindenberg"
)

# Scene / case
case = "c3sar_exp024_DOM03_v01_20260606"

# File paths
file_1d = data_dir / f"{case}_1D.nc"
file_3d = data_dir / f"{case}_3D.nc"

# Load datasets
ds_1d = xr.load_dataset(file_1d).squeeze("exp", drop=True)
ds_3d = xr.load_dataset(file_3d).squeeze("exp", drop=True)

In [ ]:
print("1D:")
ds_1d



In [ ]:
print("3D:")
ds_3d

In [ ]:
eglo_1d = ds_1d["eglo"]
eglo_3d = ds_3d["eglo"]

edn_1d = ds_1d["edn"]
edn_3d = ds_3d["edn"]

edir_1d = ds_1d["edir"]
edir_3d = ds_3d["edir"]

edir_1d = edir_1d.mean(dim="run")
edir_3d = edir_3d.mean(dim="run")

edn_1d = edn_1d.mean(dim="run")
edn_3d = edn_3d.mean(dim="run")

eglo_1d = eglo_1d.mean(dim="run")
eglo_3d = eglo_3d.mean(dim="run")

In [ ]:
eglo_1d

In [ ]:
# Data Familiarization

In [ ]:
print(ds_1d.dims)
print(ds_1d.coords)

print(ds_1d["time"].values)
print(ds_1d["sza"].values)
print(ds_1d["saa"].values)

In [ ]:
#plt solar angels over time 

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax[0].plot(ds_1d.time, ds_1d.sza)
ax[0].set_ylabel("SZA [°]")
ax[0].grid()

ax[1].plot(ds_1d.time, ds_1d.saa)
ax[1].set_ylabel("SAA [°]")
ax[1].set_xlabel("Time")
ax[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
time_indices = [5, 20, 35]

for i in time_indices:
    print(
        i,
        ds_1d.time.isel(time=i).values,
        "SZA =", float(ds_1d.sza.isel(time=i)),
        "SAA =", float(ds_1d.saa.isel(time=i))
    )

In [ ]:
time_indices = [0,1,2,3,4,5, 15,20, 25, 35]


fig, axes = plt.subplots(
    len(time_indices), 3,
    figsize=(15, 4 * len(time_indices))
)

for row, ti in enumerate(time_indices):

    e1 = edir_1d.isel(time=ti).sel(zlev=0)
    e3 = edir_3d.isel(time=ti).sel(zlev=0)
    diff = e3 - e1

    axes[row, 0].pcolormesh(
        e1.lon, e1.lat, e1,
        shading="auto"
    )

    axes[row, 1].pcolormesh(
        e3.lon, e3.lat, e3,
        shading="auto"
    )

    axes[row, 2].pcolormesh(
        diff.lon, diff.lat, diff,
        shading="auto",
        cmap="RdBu_r"
    )

    sza = float(ds_1d.sza.isel(time=ti))
    saa = float(ds_1d.saa.isel(time=ti))

    # Get actual time and format as HH:MM
    daytime = np.datetime_as_string(
        ds_1d.time.isel(time=ti).values,
        unit="m"
    ).split("T")[1]

    axes[row, 0].set_ylabel(
        f"t={ti}\n"
        f"Time={daytime}\n"
        f"SZA={sza:.1f}°\n"
        f"SAA={saa:.1f}°"
    )

axes[0, 0].set_title("1D direct")
axes[0, 1].set_title("3D direct")
axes[0, 2].set_title("3D − 1D")

plt.tight_layout()
plt.show()

In [ ]:
# shadow displacement using circular cross-correlation for one timestamp

In [ ]:
ti = 20

e1 = edir_1d.isel(time=ti).sel(zlev=0)
e3 = edir_3d.isel(time=ti).sel(zlev=0)

a = e1.values
b = e3.values

In [ ]:
sza = float(ds_1d.sza.isel(time=ti))
saa = float(ds_1d.saa.isel(time=ti))
time = ds_1d.time.isel(time=ti).values

print("Time:", time)
print(f"SZA: {sza:.2f}°")
print(f"SAA: {saa:.2f}°")




In [ ]:
# remove domain mean before correlating
a_anom = a - np.nanmean(a)
b_anom = b - np.nanmean(b)


# compute circular cross-correlation
fa = np.fft.fft2(a_anom)
fb = np.fft.fft2(b_anom)

corr = np.fft.ifft2(
    fb * np.conj(fa)
).real

# find maximum
iy, ix = np.unravel_index(
    np.nanargmax(corr),
    corr.shape
)

# convert the FFT indecies into singed peridic shifts
ny, nx = a.shape

shift_y = iy if iy <= ny // 2 else iy - ny
shift_x = ix if ix <= nx // 2 else ix - nx

print(f"Shift x: {shift_x} pixels")
print(f"Shift y: {shift_y} pixels")

In [ ]:
# apply shift and visually validate

In [ ]:
#apply shift
e1_shifted = np.roll(
    a,
    shift=(shift_y, shift_x),
    axis=(0, 1)
)

In [ ]:
# calculate RMSE
rmse_before = np.sqrt(
    np.nanmean((a - b)**2)
)

rmse_after = np.sqrt(
    np.nanmean((e1_shifted - b)**2)
)

print(f"RMSE before shift: {rmse_before:.2f} W/m²")
print(f"RMSE after shift:  {rmse_after:.2f} W/m²")

In [ ]:
#plot
fig, axes = plt.subplots(
    2, 2,
    figsize=(12, 10),
    constrained_layout=True
)

vmin = min(
    np.nanmin(a),
    np.nanmin(b)
)

vmax = max(
    np.nanmax(a),
    np.nanmax(b)
)

# 1D
im0 = axes[0, 0].pcolormesh(
    e1.lon,
    e1.lat,
    a,
    shading="auto",
    vmin=vmin,
    vmax=vmax
)

axes[0, 0].set_title("1D direct")

# 3D
axes[0, 1].pcolormesh(
    e3.lon,
    e3.lat,
    b,
    shading="auto",
    vmin=vmin,
    vmax=vmax
)

axes[0, 1].set_title("3D direct")

# Difference before
diff_before = b - a

lim = np.nanmax(
    np.abs(
        np.concatenate([
            diff_before.ravel(),
            (b - e1_shifted).ravel()
        ])
    )
)

im2 = axes[1, 0].pcolormesh(
    e1.lon,
    e1.lat,
    diff_before,
    shading="auto",
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim
)

axes[1, 0].set_title(
    f"3D − 1D\n"
    f"RMSE={rmse_before:.1f} W/m²"
)

# Difference after
diff_after = b - e1_shifted

axes[1, 1].pcolormesh(
    e1.lon,
    e1.lat,
    diff_after,
    shading="auto",
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim
)

axes[1, 1].set_title(
    f"3D − shifted 1D\n"
    f"RMSE={rmse_after:.1f} W/m²\n"
    f"Δx={shift_x}px, Δy={shift_y}px"
)

for ax in axes.flat:
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

fig.colorbar(
    im0,
    ax=axes[0, :],
    label="Direct irradiance [W m$^{-2}$]"
)

fig.colorbar(
    im2,
    ax=axes[1, :],
    label="Difference [W m$^{-2}$]"
)

plt.show()

In [ ]:
#convert shift into meters
lat_mean = float(e1.lat.mean())

dx = (
    np.mean(np.diff(e1.lon.values))
    * 111_320
    * np.cos(np.deg2rad(lat_mean))
)

dy = (
    np.mean(np.diff(e1.lat.values))
    * 111_320
)


shift_x_m = shift_x * dx
shift_y_m = shift_y * dy

shift_distance = np.sqrt(
    shift_x_m**2 +
    shift_y_m**2
)

print(f"dx = {dx:.1f} m")
print(f"dy = {dy:.1f} m")

print(f"x displacement = {shift_x_m:.1f} m")
print(f"y displacement = {shift_y_m:.1f} m")
print(f"total displacement = {shift_distance:.1f} m")

In [ ]:
# calculate displacemnt direction
# 0° = north
# 90° = east
# 180° = south
# 270° = west

shift_direction = (
    np.degrees(
        np.arctan2(
            shift_x_m,
            shift_y_m
        )
    ) % 360
)

print(
    f"Displacement direction: "
    f"{shift_direction:.1f}°"
)


In [ ]:
# get solar geometry

sza = float(
    ds_1d.sza.isel(time=ti)
)

saa = float(
    ds_1d.saa.isel(time=ti)
)

time = ds_1d.time.isel(time=ti).values

print(f"Time: {time}")
print(f"SZA: {sza:.2f}°")
print(f"SAA: {saa:.2f}°")
print(f"Shift direction: {shift_direction:.2f}°")

In [ ]:
# turn that into a function

In [ ]:
def get_shadow_shift(
    edir_1d,
    edir_3d,
    ds,
    time_index,
):

    e1 = (
        edir_1d
        .isel(time=time_index)
        .sel(zlev=0)
    )

    e3 = (
        edir_3d
        .isel(time=time_index)
        .sel(zlev=0)
    )

    a = e1.values
    b = e3.values

    sza = float(ds.sza.isel(time=time_index))
    saa = float(ds.saa.isel(time=time_index))
    time = ds.time.isel(time=time_index).values

    # ------------------------------------------------------------
    # Missing data
    # ------------------------------------------------------------
    if not np.isfinite(a).any() or not np.isfinite(b).any():

        return {
            "time_index": time_index,
            "time": time,
            "sza": sza,
            "saa": saa,
            "valid": False,
            "shift_x_px": np.nan,
            "shift_y_px": np.nan,
            "shift_x_m": np.nan,
            "shift_y_m": np.nan,
            "displacement_m": np.nan,
            "direction_deg": np.nan,
            "rmse_before": np.nan,
            "rmse_after": np.nan
        }

    # ------------------------------------------------------------
    # Remove mean
    # ------------------------------------------------------------
    a_anom = a - np.mean(a)
    b_anom = b - np.mean(b)

    # Circular FFT cross-correlation
    fa = np.fft.fft2(a_anom)
    fb = np.fft.fft2(b_anom)

    corr = np.fft.ifft2(
        fb * np.conj(fa)
    ).real

    iy, ix = np.unravel_index(
        np.argmax(corr),
        corr.shape
    )

    ny, nx = a.shape

    shift_y = iy if iy <= ny // 2 else iy - ny
    shift_x = ix if ix <= nx // 2 else ix - nx

    # ------------------------------------------------------------
    # Grid spacing
    # ------------------------------------------------------------
    lat_mean = float(e1.lat.mean())

    dx = (
        np.mean(np.diff(e1.lon.values))
        * 111_320
        * np.cos(np.deg2rad(lat_mean))
    )

    dy = (
        np.mean(np.diff(e1.lat.values))
        * 111_320
    )


    shift_x_m = shift_x * dx
    shift_y_m = shift_y * dy

    displacement = np.sqrt(
        shift_x_m**2 + shift_y_m**2
    )

    direction = (
        np.degrees(
            np.arctan2(
                shift_x_m,
                shift_y_m
            )
        ) % 360
    )

    # Shifted field
    shifted = np.roll(
        a,
        shift=(shift_y, shift_x),
        axis=(0, 1)
    )

    # RMSE
    rmse_before = np.sqrt(
        np.mean((a - b)**2)
    )

    rmse_after = np.sqrt(
        np.mean((shifted - b)**2)
    )

    return {
        "time_index": time_index,
        "time": time,
        "sza": sza,
        "saa": saa,
        "valid": True,
        "shift_x_px": shift_x,
        "shift_y_px": shift_y,
        "shift_x_m": shift_x_m,
        "shift_y_m": shift_y_m,
        "displacement_m": displacement,
        "direction_deg": direction,
        "rmse_before": rmse_before,
        "rmse_after": rmse_after
    }

In [ ]:
#test the function

result = get_shadow_shift(
    edir_1d,
    edir_3d,
    ds_1d,
    time_index=20
)

result

In [ ]:
# apply to all timestamps


In [ ]:
results = []

for ti in range(ds_1d.sizes["time"]):
    
    result = get_shadow_shift(
        edir_1d,
        edir_3d,
        ds_1d,
        time_index=ti
    )
    
    results.append(result)

In [ ]:
# convert into dataframe
df_shift = pd.DataFrame(results)

df_shift


In [ ]:
# inspect invalid timestamps
df_shift[~df_shift["valid"]]

In [ ]:
#remove invalid timestamps
df_shift = df_shift[df_shift["valid"]].copy()

In [ ]:
# first plots

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df_shift["sza"],
    df_shift["displacement_m"]
)

plt.xlabel("Solar zenith angle [°]")
plt.ylabel("Shadow displacement [m]")

plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    np.tan(np.deg2rad(df_shift["sza"])),
    df_shift["displacement_m"]
)

plt.xlabel("tan(SZA)")
plt.ylabel("Shadow displacement [m]")
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df_shift["saa"],
    df_shift["direction_deg"]
)

plt.xlabel("Solar azimuth angle [°]")
plt.ylabel("Shadow displacement direction [°]")

plt.grid()

plt.show()

In [ ]:
fig, axes = plt.subplots(
    4, 1,
    figsize=(10, 10),
    sharex=True
)

axes[0].plot(
    df_shift["time"],
    df_shift["displacement_m"]
)
axes[0].set_ylabel("Displacement [m]")

axes[1].plot(
    df_shift["time"],
    df_shift["shift_x_px"],
    label="x"
)
axes[1].plot(
    df_shift["time"],
    df_shift["shift_y_px"],
    label="y"
)
axes[1].set_ylabel("Shift [px]")
axes[1].legend()

axes[2].plot(
    df_shift["time"],
    df_shift["rmse_before"],
    label="before"
)
axes[2].plot(
    df_shift["time"],
    df_shift["rmse_after"],
    label="after"
)
axes[2].set_ylabel("RMSE [W m$^{-2}$]")
axes[2].legend()

# Solar angles
ax_sza = axes[3]

ax_sza.plot(
    df_shift["time"],
    df_shift["sza"],
    label="SZA"
)
ax_sza.set_ylabel("SZA [°]")
ax_sza.set_xlabel("Time")

ax_saa = ax_sza.twinx()

ax_saa.plot(
    df_shift["time"],
    df_shift["saa"],
    linestyle="--",
    label="SAA"
)
ax_saa.set_ylabel("SAA [°]")

# Combined legend
lines1, labels1 = ax_sza.get_legend_handles_labels()
lines2, labels2 = ax_saa.get_legend_handles_labels()

ax_sza.legend(
    lines1 + lines2,
    loc="best"
)

plt.tight_layout()
plt.show()

In [ ]:
#check for suspiciouls timestamps

In [ ]:
ti_suspicious = 3


In [ ]:
time_indices = [
    ti_suspicious - 1,
    ti_suspicious,
    ti_suspicious + 1
]

df_shift[
    df_shift["time_index"].isin(time_indices)
]

In [ ]:
ti = ti_suspicious
run_index = 0

e1 = (
    edir_1d
    .isel(time=ti)
    .sel(zlev=0)
)

e3 = (
    edir_3d
    .isel(time=ti)
    .sel(zlev=0)
)

a = e1.values
b = e3.values

row = df_shift.loc[
    df_shift["time_index"] == ti
].iloc[0]

shift_x = int(row["shift_x_px"])
shift_y = int(row["shift_y_px"])

e1_shifted = np.roll(
    a,
    shift=(shift_y, shift_x),
    axis=(0, 1)
)

diff_before = b - a
diff_after = b - e1_shifted

rmse_before = np.sqrt(
    np.nanmean((a - b)**2)
)

rmse_after = np.sqrt(
    np.nanmean((e1_shifted - b)**2)
)

fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 9),
    constrained_layout=True
)

vmin = min(
    np.nanmin(a),
    np.nanmin(b),
    np.nanmin(e1_shifted)
)

vmax = max(
    np.nanmax(a),
    np.nanmax(b),
    np.nanmax(e1_shifted)
)

lim = np.nanmax(
    np.abs(
        np.concatenate([
            diff_before.ravel(),
            diff_after.ravel()
        ])
    )
)

# radiation fields
im0 = axes[0, 0].pcolormesh(
    e1.lon, e1.lat, a,
    shading="auto",
    vmin=vmin,
    vmax=vmax
)
axes[0, 0].set_title("1D direct")

axes[0, 1].pcolormesh(
    e3.lon, e3.lat, b,
    shading="auto",
    vmin=vmin,
    vmax=vmax
)
axes[0, 1].set_title("3D direct")

axes[0, 2].pcolormesh(
    e1.lon, e1.lat, e1_shifted,
    shading="auto",
    vmin=vmin,
    vmax=vmax
)
axes[0, 2].set_title(
    f"Shifted 1D\n"
    f"Δx={shift_x}px, Δy={shift_y}px"
)

# differences
im1 = axes[1, 0].pcolormesh(
    e1.lon, e1.lat, diff_before,
    shading="auto",
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim
)
axes[1, 0].set_title(
    f"3D − 1D\n"
    f"RMSE={rmse_before:.1f} W m$^{{-2}}$"
)

axes[1, 1].pcolormesh(
    e1.lon, e1.lat, diff_after,
    shading="auto",
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim
)
axes[1, 1].set_title(
    f"3D − shifted 1D\n"
    f"RMSE={rmse_after:.1f} W m$^{{-2}}$"
)

axes[1, 2].axis("off")

for ax in axes.flat:
    if ax.has_data():
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_aspect("equal")

fig.colorbar(
    im0,
    ax=axes[0, :],
    label="Direct irradiance [W m$^{-2}$]"
)

fig.colorbar(
    im1,
    ax=axes[1, :2],
    label="Difference [W m$^{-2}$]"
)

sza = float(ds_1d.sza.isel(time=ti))
saa = float(ds_1d.saa.isel(time=ti))

daytime = np.datetime_as_string(
    ds_1d.time.isel(time=ti).values,
    unit="m"
).split("T")[1]

fig.suptitle(
    f"{daytime} UTC | "
    f"SZA={sza:.1f}° | "
    f"SAA={saa:.1f}°"
)

plt.show()

first few timestpams where we have nearly no cloud seem to be the problem
the clean conceptual fix is to introduce a cloudiness / shadow-content criterion before accepting a shift. 
In other words: only estimate shadow displacement when there is enough cloud-induced structure in edir for the cross-correlation to be meaningful.

In [ ]:
# Early timestamps are excluded from the displacement analysis because cloud cover is very low and the direct-radiation field contains insufficient spatial structure for a robust FFT-based shift estimate. 
# A cloudiness or shadow-content quality criterion will be introduced later for automated filtering.


# Exclude first four timestamps and invalid entries
df_plot = df_shift[
    (df_shift["time_index"] >= 4) &
    (df_shift["valid"])
].copy()

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5),
    constrained_layout=True
)

# ============================================================
# Left: displacement vs tan(SZA)
# ============================================================

x1 = np.tan(np.deg2rad(df_plot["sza"]))
y1 = df_plot["displacement_m"]

mask1 = np.isfinite(x1) & np.isfinite(y1)

axes[0].scatter(
    x1[mask1],
    y1[mask1],
    color="tab:blue",
    label="Data"
)

coef1 = np.polyfit(
    x1[mask1],
    y1[mask1],
    1
)

x1_fit = np.linspace(
    x1[mask1].min(),
    x1[mask1].max(),
    100
)

y1_fit = np.polyval(
    coef1,
    x1_fit
)

axes[0].plot(
    x1_fit,
    y1_fit,
    color="tab:red",
    linewidth=2,
    label="Linear fit"
)

axes[0].set_xlabel("tan(SZA)")
axes[0].set_ylabel("Shadow displacement [m]")
axes[0].set_title("Displacement magnitude")
axes[0].grid()
axes[0].legend()


# ============================================================
# Right: displacement direction vs SAA
# ============================================================

x2 = df_plot["saa"].values
y2 = df_plot["direction_deg"].values

mask2 = np.isfinite(x2) & np.isfinite(y2)

axes[1].scatter(
    x2[mask2],
    y2[mask2],
    color="tab:blue",
    label="Data"
)



# 1:1 reference line
axes[1].plot(
    [0, 360],
    [0, 360],
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="1:1 line"
)

axes[1].set_xlim(0, 360)
axes[1].set_ylim(0, 360)

axes[1].set_xlabel("Solar azimuth angle [°]")
axes[1].set_ylabel("Shadow displacement direction [°]")
axes[1].set_title("Displacement direction")
axes[1].grid()
axes[1].legend()

plt.show()

In [ ]:
# Exclude first four timestamps and invalid entries
df_plot = df_shift[
    (df_shift["time_index"] >= 4) &
    (df_shift["valid"])
].copy()

# Data
x = np.tan(np.deg2rad(df_plot["sza"]))
y = df_plot["displacement_m"]

mask = np.isfinite(x) & np.isfinite(y)

# Linear fit
coef = np.polyfit(
    x[mask],
    y[mask],
    1
)

x_fit = np.linspace(
    x[mask].min(),
    x[mask].max(),
    100
)

y_fit = np.polyval(
    coef,
    x_fit
)

# Plot
plt.figure(figsize=(7, 5))

plt.scatter(
    x[mask],
    y[mask],
    color="tab:blue",
    label="Data"
)

plt.plot(
    x_fit,
    y_fit,
    color="tab:red",
    linewidth=2,
    label="Linear fit"
)

plt.xlabel("tan(SZA)")
plt.ylabel("Shadow displacement [m]")
plt.title("Shadow displacement vs. solar zenith angle")

plt.grid()
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# use fit to get clouuf hight

ordinary linear fit:
$$d= h * tan(SZA)+ b$$

fit forced through the origin:
physically more directly interpretable because if SZA → 0, the geometric horizontal shadow displacement should also approach 0.
$$ d=h_{eff}*tan(SZA) $$

In [ ]:
# Exclude first four timestamps and invalid entries
df_plot = df_shift[
    (df_shift["time_index"] >= 4) &
    (df_shift["valid"])
].copy()

x = np.tan(np.deg2rad(df_plot["sza"])).values
y = df_plot["displacement_m"].values

mask = np.isfinite(x) & np.isfinite(y)

x = x[mask]
y = y[mask]


# ============================================================
# 1. Ordinary linear fit: y = a*x + b
# ============================================================

a, b = np.polyfit(x, y, 1)

y_fit_linear = a * x + b


# ============================================================
# 2. Fit forced through origin: y = h_eff*x
# ============================================================

h_eff = np.sum(x * y) / np.sum(x**2)

y_fit_origin = h_eff * x


# ============================================================
# R² values
# ============================================================

ss_tot = np.sum((y - np.mean(y))**2)

r2_linear = 1 - np.sum((y - y_fit_linear)**2) / ss_tot
r2_origin = 1 - np.sum((y - y_fit_origin)**2) / ss_tot


# ============================================================
# Print results
# ============================================================

print("Ordinary linear fit:")
print(f"  slope     = {a:.1f} m")
print(f"  intercept = {b:.1f} m")
print(f"  R²        = {r2_linear:.3f}")

print("\nFit through origin:")
print(f"  effective cloud height = {h_eff:.1f} m")
print(f"  R²                     = {r2_origin:.3f}")

In [ ]:
x_fit = np.linspace(
    x.min(),
    x.max(),
    100
)

plt.figure(figsize=(7, 5))

plt.scatter(
    x,
    y,
    color="tab:blue",
    label="Data"
)

plt.plot(
    x_fit,
    a * x_fit + b,
    color="tab:red",
    linewidth=2,
    label=f"Linear fit: y = {a:.0f}x + {b:.0f}"
)

plt.plot(
    x_fit,
    h_eff * x_fit,
    color="tab:green",
    linestyle="--",
    linewidth=2,
    label=f"Through origin: h = {h_eff:.0f} m"
)

plt.xlabel("tan(SZA)")
plt.ylabel("Shadow displacement [m]")
plt.title("Shadow displacement vs. solar zenith angle")

plt.grid()
plt.legend()

plt.tight_layout()
plt.show()

### Check h_eff for every timetamp


In [ ]:
df_shift["h_eff_m"] = (
    df_shift["displacement_m"]
    / np.tan(np.deg2rad(df_shift["sza"]))
)

In [ ]:
df_plot = df_shift[
    (df_shift["time_index"] >= 4) &
    (df_shift["valid"])
].copy()

df_plot[
    [
        "time",
        "sza",
        "displacement_m",
        "h_eff_m"
    ]
]

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    df_plot["time"],
    df_plot["displacement_m"],
    marker="o"
)

plt.xlabel("Time")
plt.ylabel("Shadow displacement [m]")
plt.title("Shadow displacement vs SZA")

plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

# Shadow displacement
ax1.plot(
    df_plot["time"],
    df_plot["displacement_m"],
    marker="o",
    label="Shadow displacement"
)

ax1.set_xlabel("Time")
ax1.set_ylabel("Shadow displacement [m]")
ax1.grid()

# Second y-axis for SZA
ax2 = ax1.twinx()

ax2.plot(
    df_plot["time"],
    df_plot["sza"],
    linestyle="--",
    label="SZA"
)

ax2.set_ylabel("Solar zenith angle [°]")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="best"
)

plt.title("Shadow displacement and solar zenith angle")

fig.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    df_plot["time"],
    df_plot["h_eff_m"],
    marker="o"
)

plt.xlabel("Time")
plt.ylabel("Effective shadow height [m]")
plt.title("Effective shadow displacement height")

plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    2, 1,
    figsize=(9, 8),
    sharex=True,
    constrained_layout=True
)

# ------------------------------------------------------------
# Top: displacement + SZA
# ------------------------------------------------------------
ax1 = axes[0]

ax1.plot(
    df_plot["time"],
    df_plot["displacement_m"],
    marker="o",
    label="Shadow displacement"
)

ax1.set_ylabel("Shadow displacement [m]")
ax1.grid()

ax1b = ax1.twinx()

ax1b.plot(
    df_plot["time"],
    df_plot["sza"],
    linestyle="--",
    label="SZA"
)

ax1b.set_ylabel("Solar zenith angle [°]")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="best"
)

ax1.set_title("Shadow displacement and solar zenith angle")


# ------------------------------------------------------------
# Bottom: effective height
# ------------------------------------------------------------
axes[1].plot(
    df_plot["time"],
    df_plot["h_eff_m"],
    marker="o",
    label=r"$h_{\mathrm{eff}} = \frac{d}{\tan(\mathrm{SZA})}$"
)

axes[1].set_xlabel("Time")
axes[1].set_ylabel("Effective shadow height [m]")
axes[1].set_title("Effective shadow displacement height")

axes[1].grid()
axes[1].legend()

plt.show()

In [ ]:
print(
    f"Mean h_eff:   {df_plot['h_eff_m'].mean():.1f} m"
)

print(
    f"Median h_eff: {df_plot['h_eff_m'].median():.1f} m"
)

print(
    f"Std h_eff:    {df_plot['h_eff_m'].std():.1f} m"
)

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df_plot["sza"],
    df_plot["h_eff_m"]
)

plt.xlabel("Solar zenith angle [°]")
plt.ylabel("Effective shadow height [m]")
plt.title("Effective height vs. solar zenith angle")

plt.grid()
plt.tight_layout()
plt.show()